In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
from torch.utils.data import DataLoader, TensorDataset

    
        
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
from data_loading import *
from data_generation import *
from data_plot import *
from approaches.standardized_residuals import StandardizedResiduals
from models import *

In [ ]:
torch.manual_seed(42)

N = 2000
dim_X = 1
dim_y = 2

X_train = torch.ones(N, dim_X)
Y_train = torch.empty(N, dim_y).exponential_(1.0)

X_calibration = torch.ones(N, dim_X)
Y_calibration = torch.empty(N, dim_y).exponential_(1.0)

tau = 0.70        
batch_size = 200
num_epochs = 1000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
torch.manual_seed(42)

generator = Generator2D(
    f=circle_f,
    matrix_transform=strange_matrix_transform,
    noise_std=1.0,
    noise_type="exponential"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 2

tau = 0.95

batch_size = 256
num_epochs = 1_000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")


standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)

plot_samples_with_contours_multi_flow(generator, model, tau=tau, num_slices=10)
plot_chdr_with_samples_gaussian(standardized_residuals, generator, num_slices=10, x_range=(-1, 1), name="exponential")
# plot_2d_conditional_contours(generator, model, xs=[-0.8, -0.4, 0.0, 0.4, 0.8])
plot_combined_conditional_contours_multiflow(generator, standardized_residuals, model)

In [ ]:
torch.manual_seed(42)

N = 2000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)

# Y suit une loi exponentielle 2D indépendante
outlier_fraction = 0.1
num_outliers = int(N * outlier_fraction)

Y_train = torch.empty(N, dim_y).normal_(mean=40.0, std=1.0)
outliers = torch.empty(num_outliers, dim_y).normal_(mean=10.0, std=5.0)
indices = torch.randperm(N)[:num_outliers]
Y_train[indices] = outliers

Y_train = ( Y_train - torch.mean(Y_train) ) / torch.std(Y_train)


X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%
batch_size = 250
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )


model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
import numpy as np

def generate_gaussian_mixture(N):
    # Mixture weights (must sum to 1)
    weights = np.array([0.5, 0.5])
    
    # Means of the 4 Gaussians (2D)
    means = np.array([
        [-3, -3],
        [3, 3],
    ])
    
    # Covariance matrices
    covs = [
        [[0.5, 0], [0, 0.5]],
        [[0.5, 0], [0, 0.5]],
    ]

    # Choose which Gaussian each sample comes from
    components = np.random.choice(2, size=N, p=weights)

    Y_train = np.zeros((N, 2))

    for k in range(2):
        idx = components == k
        n_k = np.sum(idx)
        if n_k > 0:
            Y_train[idx] = np.random.multivariate_normal(means[k], covs[k], n_k)

    return torch.as_tensor(Y_train, dtype=torch.float32)


N = 2000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)
Y_train = generate_gaussian_mixture(N)


X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%
lambda_val = 10
batch_size = 256
num_epochs = 1_000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
import torch
import numpy as np

def generate_full_star(N, dim_y=2, sigma=0.8):
    points = []
    batch_size = N * 3 
    
    while len(points) < N:
        # MODIFIED: Propose points from a 2D Gaussian centered at (0,0)
        # sigma controls how tightly points cluster in the center of the star
        candidate_points = torch.randn(batch_size, 2) * sigma
        
        r = torch.sqrt(candidate_points[:, 0]**2 + candidate_points[:, 1]**2)
        theta = torch.atan2(candidate_points[:, 1], candidate_points[:, 0])
        
        star_radius = 1.0 + 0.8 * torch.cos(5 * theta / 2).abs() 
        
        mask = r < star_radius
        accepted = candidate_points[mask]
        points.append(accepted)
        
    Y_star = torch.cat(points, dim=0)[:N]
    
    if dim_y > 2:
        extra = torch.randn(N, dim_y - 2) * 0.01
        Y_star = torch.cat([Y_star, extra], dim=1)
        
    return Y_star

def generate_star(N, dim_y=2, loc=5.0, scale=2.0):
    if dim_y < 2:
        raise ValueError("Star shape requires at least 2 dimensions.")
    
    # Angles for the 5 outer points and 5 inner points of a star
    angles = np.linspace(0, 2 * np.pi, 11)
    r_outer = 2.0
    r_inner = 0.8
    
    radii = np.ones_like(angles)
    radii[::2] = r_outer
    radii[1::2] = r_inner
    
    pts_x = radii * np.cos(angles)
    pts_y = radii * np.sin(angles)
    
    # MODIFIED: Sample 't' from a Gaussian instead of linspace
    # loc=5.0 centers the cluster at the bottom point of the star
    t = np.random.normal(loc=loc, scale=scale, size=N)
    t = np.mod(t, 10.0) # Wrap around so points stay on the 10 edges
    
    star_x = np.interp(t, np.arange(11), pts_x)
    star_y = np.interp(t, np.arange(11), pts_y)
    
    star_data = np.stack([star_x, star_y], axis=1)
    star_tensor = torch.from_numpy(star_data).float()
    star_tensor += torch.randn_like(star_tensor) * 0.05 # Add jitter
    
    if dim_y > 2:
        extra = torch.zeros(N, dim_y - 2)
        star_tensor = torch.cat([star_tensor, extra], dim=1)
        
    return star_tensor

In [ ]:
torch.manual_seed(42)


N = 2000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)
Y_train = generate_full_star(N, dim_y)


X_calibration = X_train
Y_calibration = Y_train


tau = 0.98        # Cible de couverture : tau%

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
torch.manual_seed(42)

N = 2000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)
Y_train = generate_star(N, dim_y)


X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%
lambda_val = 1.
lr_lambda = 0.0
batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
torch.manual_seed(42)

N = 2000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)
Y_train = sample_annulus(N, 9.5, 10.0)


X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
class GeneratorD:
    def __init__(self, f, matrix_transform, dim_y, noise_type='gaussian', noise_std=1.0):
        self.f = f
        self.matrix_transform = matrix_transform
        self.noise_type = noise_type
        self.noise_std = noise_std
        self.dim_y = dim_y

    def _get_noise(self, n):
        if self.noise_type == 'gaussian':
            noise = torch.randn(n, self.dim_y)
        elif self.noise_type == 'uniform':
            noise = torch.rand(n, self.dim_y) * 2 - 1
        elif self.noise_type == 'exponential':
            # noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y))
            noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
        elif self.noise_type == 'multimodal':
            # noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y))
            noise1 = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
            noise2 = - torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 3.0
            mask = (torch.rand(n, 1) > 0.5).float()
            noise = mask * noise1 + (1.0 - mask) * noise2
        else:
            raise ValueError(f"Type inconnu: {self.noise_type}")
        return (noise * self.noise_std).unsqueeze(2)

    def generate(self, n):
        x = 2 * torch.rand(n, 1) - 1
        fx = self.f(x)
        A_x = self.matrix_transform(x)
        noise = self._get_noise(n)
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y = fx + correlated_noise
        return x, y

    def generate_specific_y_given_x(self, x_tensor, n=1):
        x_repeated = x_tensor.repeat_interleave(n, dim=0)
        fx = self.f(x_repeated)
        A_x = self.matrix_transform(x_repeated)
        noise = self._get_noise(x_repeated.shape[0])
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y_flat = fx + correlated_noise
        return y_flat.view(n, self.dim_y)

def strange_matrix_transform_1D(x):
    n = x.shape[0]
    matrices = torch.eye(1).unsqueeze(0).repeat(n, 1, 1)
    matrices[:, 0, 0] = x.squeeze(-1)**2 + 0.5 
    return matrices

def circle_f_1D(x):
    return torch.sin(x*3)



# ==========================================
# 3. Entraînement
# ==========================================
torch.manual_seed(42)

generator = GeneratorD(
    f=circle_f_1D,
    matrix_transform=strange_matrix_transform_1D,
    dim_y=1,
    noise_std=1.0,
    noise_type="exponential"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_val, Y_val = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 1


In [ ]:
tau = 0.80      

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)

In [ ]:
tau = 0.3    

batch_size = 256
num_epochs = 1_000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D)

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)

In [ ]:
tau = 0.95

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D)

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)

In [ ]:
# ==========================================
# 3. Entraînement
# ==========================================
torch.manual_seed(42)

generator = GeneratorD(
    f=circle_f_1D,
    matrix_transform=strange_matrix_transform_1D,
    dim_y=1,
    noise_std=1.0,
    noise_type="multimodal"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_val, Y_val = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 1


In [ ]:
tau = 0.80      

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)

In [ ]:
tau = 0.3    

batch_size = 256
num_epochs = 1_000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=5, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D)

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)

In [ ]:
tau = 0.95

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D)

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_test)

print("Average Volume:", volumes)